## Notebook 02 — RibFrac CT Preprocessing
<p> Patient-level splitting is performed in Notebook 01 and is preserved
throughout the preprocessing pipeline.</p>
<p>This notebook preprocesses RibFrac CT volumes and their corresponding
fracture annotation volumes for the Enhanced 3D ResUNet framework.</p>
<p> The main goal of Notebook 02 is to transform each CT volume and its corresponding annotation into a consistent 3D representation suitable for the Enhanced 3D ResUNet.</p>

<h4>Processing includes:</h4>
<ol>
<li>CT and label loading</li>
<li> Spatial validation</li>
<li> Orientation standardization</li>
<li> CT intensity clipping</li>
<li>CT intensity normalization</li>
<li> Spatial resampling</li>
<li> Label resampling using nearest-neighbor interpolation</li>
<li> Preprocessing validation</li>
<li> Saving the processed volumes</li>
</ol>


In [ ]:
#3. Define directories
## We will use the directories established in Notebook 01.

BASE_DIR = Path("/kaggle/working")

IMAGE_DIR = BASE_DIR / "ribfrac_images"
LABEL_DIR = BASE_DIR / "ribfrac_labels"

PREPROCESSED_DIR = BASE_DIR / "preprocessed"

PREPROCESSED_IMAGE_DIR = PREPROCESSED_DIR / "images"
PREPROCESSED_LABEL_DIR = PREPROCESSED_DIR / "labels"

PREPROCESSED_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
PREPROCESSED_LABEL_DIR.mkdir(parents=True, exist_ok=True)

print("Preprocessing directories created:")
print(PREPROCESSED_IMAGE_DIR)
print(PREPROCESSED_LABEL_DIR)

In [ ]:
# 4. — Check directories
print("Image directory exists:", IMAGE_DIR.exists())
print("Label directory exists:", LABEL_DIR.exists())

print("\nPreprocessed directories:")
print("Images:", PREPROCESSED_IMAGE_DIR)
print("Labels:", PREPROCESSED_LABEL_DIR)

In [ ]:
#5. — Define preprocessing parameters
HU_MIN = -1000.0
HU_MAX = 400.0

TARGET_SPACING = (1.0, 1.0, 1.0)

print("HU range:", HU_MIN, "to", HU_MAX)
print("Target spacing:", TARGET_SPACING)

In [ ]:
#6. — Load a patient
def load_patient_for_preprocessing(patient_id):
    """
    Load one patient's CT and label volumes.

    The CT retrieval function and label retrieval function
    are defined in Notebook 01.
    """

    ct_path = download_ribfrac_image(patient_id)
    label_path = get_label_path(patient_id)

    ct_img = nib.load(str(ct_path))
    label_img = nib.load(str(label_path))

    ct_volume = ct_img.get_fdata(dtype=np.float32)
    label_volume = label_img.get_fdata(dtype=np.float32)

    return {
        "patient_id": patient_id,
        "ct_path": ct_path,
        "label_path": label_path,
        "ct_img": ct_img,
        "label_img": label_img,
        "ct_volume": ct_volume,
        "label_volume": label_volume
    }

In [ ]:
#7 — Spatial validation
def validate_spatial_alignment(ct_img, label_img):
    """
    Validate shape, voxel spacing, and affine alignment.
    """

    ct_shape = ct_img.shape
    label_shape = label_img.shape

    ct_spacing = ct_img.header.get_zooms()[:3]
    label_spacing = label_img.header.get_zooms()[:3]

    shape_match = ct_shape == label_shape
    spacing_match = np.allclose(ct_spacing, label_spacing, atol=1e-5)
    affine_match = np.allclose(ct_img.affine, label_img.affine, atol=1e-4)

    return {
        "shape_match": shape_match,
        "spacing_match": spacing_match,
        "affine_match": affine_match,
        "ct_shape": ct_shape,
        "label_shape": label_shape,
        "ct_spacing": ct_spacing,
        "label_spacing": label_spacing
    }

In [ ]:
#8 — CT intensity preprocessing
def preprocess_ct_intensity(ct_volume,
                            hu_min=HU_MIN,
                            hu_max=HU_MAX):
    """
    Clip CT intensities and normalize to [0, 1].
    """

    ct_volume = np.asarray(ct_volume, dtype=np.float32)

    # Clip HU values
    ct_volume = np.clip(ct_volume, hu_min, hu_max)

    # Normalize to [0, 1]
    ct_volume = (ct_volume - hu_min) / (hu_max - hu_min)

    return ct_volume.astype(np.float32)

In [ ]:
#9 — Resampling function
def calculate_zoom_factors(original_spacing, target_spacing):
    """
    Calculate zoom factors for spatial resampling.
    """

    original_spacing = np.asarray(original_spacing, dtype=np.float32)
    target_spacing = np.asarray(target_spacing, dtype=np.float32)

    return original_spacing / target_spacing

In [ ]:
#10 — Resample CT
def resample_ct(ct_volume,
                original_spacing,
                target_spacing=TARGET_SPACING):

    zoom_factors = calculate_zoom_factors(
        original_spacing,
        target_spacing
    )

    resampled_ct = zoom(
        ct_volume,
        zoom=zoom_factors,
        order=1
    )

    return resampled_ct.astype(np.float32)

In [ ]:
#11 — Resample label
def resample_label(label_volume,
                   original_spacing,
                   target_spacing=TARGET_SPACING):

    zoom_factors = calculate_zoom_factors(
        original_spacing,
        target_spacing
    )

    resampled_label = zoom(
        label_volume,
        zoom=zoom_factors,
        order=0
    )

    return resampled_label.astype(np.int16)

In [ ]:
#12 — Complete preprocessing function
def preprocess_patient(patient_id):

    print("=" * 60)
    print(f"Processing patient: {patient_id}")
    print("=" * 60)

    # -------------------------------------------------
    # 1. Load patient
    # -------------------------------------------------

    patient = load_patient_for_preprocessing(patient_id)

    ct_img = patient["ct_img"]
    label_img = patient["label_img"]

    ct_volume = patient["ct_volume"]
    label_volume = patient["label_volume"]

    # -------------------------------------------------
    # 2. Validate spatial alignment
    # -------------------------------------------------

    validation = validate_spatial_alignment(
        ct_img,
        label_img
    )

    if not validation["shape_match"]:
        raise ValueError(
            f"CT/label shape mismatch for {patient_id}"
        )

    if not validation["spacing_match"]:
        raise ValueError(
            f"CT/label spacing mismatch for {patient_id}"
        )

    if not validation["affine_match"]:
        raise ValueError(
            f"CT/label affine mismatch for {patient_id}"
        )

    print("Spatial validation: PASS")

    # -------------------------------------------------
    # 3. Get original spacing
    # -------------------------------------------------

    original_spacing = ct_img.header.get_zooms()[:3]

    print("Original spacing:", original_spacing)

    # -------------------------------------------------
    # 4. CT intensity preprocessing
    # -------------------------------------------------

    ct_volume = preprocess_ct_intensity(ct_volume)

    print(
        "CT intensity range after normalization:",
        float(ct_volume.min()),
        "to",
        float(ct_volume.max())
    )

    # -------------------------------------------------
    # 5. Resample CT
    # -------------------------------------------------

    ct_resampled = resample_ct(
        ct_volume,
        original_spacing,
        TARGET_SPACING
    )

    # -------------------------------------------------
    # 6. Resample label
    # -------------------------------------------------

    label_resampled = resample_label(
        label_volume,
        original_spacing,
        TARGET_SPACING
    )

    # -------------------------------------------------
    # 7. Validate output
    # -------------------------------------------------

    print("Resampled CT shape:", ct_resampled.shape)
    print("Resampled label shape:", label_resampled.shape)

    print(
        "CT range:",
        float(ct_resampled.min()),
        "to",
        float(ct_resampled.max())
    )

    print(
        "Label values:",
        np.unique(label_resampled)
    )

    return {
        "patient_id": patient_id,
        "ct": ct_resampled,
        "label": label_resampled,
        "original_spacing": original_spacing,
        "target_spacing": TARGET_SPACING
    }

In [ ]:
#13 — Test with RibFrac128
test_patient = preprocess_patient("RibFrac128")

In [ ]:
#14 — Validate label values
unique_labels = np.unique(test_patient["label"])

print("Unique label values:")
print(unique_labels)

if np.all(unique_labels == unique_labels.astype(int)):
    print("Label integrity check: PASS")
else:
    print("Label integrity check: FAIL")

In [ ]:
#15 — Check CT and label dimensions
ct_shape = test_patient["ct"].shape
label_shape = test_patient["label"].shape

print("CT shape:", ct_shape)
print("Label shape:", label_shape)

assert ct_shape == label_shape

print("CT/label shape check: PASS")

In [ ]:
#16 — Save preprocessed patient
def save_preprocessed_patient(patient_data):

    patient_id = patient_data["patient_id"]

    ct = patient_data["ct"]
    label = patient_data["label"]

    ct_path = PREPROCESSED_IMAGE_DIR / f"{patient_id}-image.npy"
    label_path = PREPROCESSED_LABEL_DIR / f"{patient_id}-label.npy"

    np.save(ct_path, ct)
    np.save(label_path, label)

    print(f"Saved CT:    {ct_path}")
    print(f"Saved label: {label_path}")

    return ct_path, label_path

In [ ]:
#18 — Verify saved files
saved_ct = np.load(ct_path)
saved_label = np.load(label_path)

print("Saved CT shape:", saved_ct.shape)
print("Saved label shape:", saved_label.shape)

print("Saved CT dtype:", saved_ct.dtype)
print("Saved label dtype:", saved_label.dtype)

In [ ]:
#19 — Check memory
print(
    "CT memory:",
    saved_ct.nbytes / (1024 ** 2),
    "MB"
)

print(
    "Label memory:",
    saved_label.nbytes / (1024 ** 2),
    "MB"
)

In [ ]:
#20 — Release test patient
del test_patient
del saved_ct
del saved_label

gc.collect()

print("Temporary arrays released from memory.")

In [ ]:
#21 — Automatic preprocessing of the training set
preprocess_patient("RibFrac128")

In [ ]:
for i, patient_id in enumerate(train_patients, start=1):

    print(f"\nPatient {i}/{len(train_patients)}")
    print(f"Patient ID: {patient_id}")

    processed = preprocess_patient(patient_id)

    save_preprocessed_patient(processed)

    # Release memory
    del processed
    gc.collect()